# Predicting Real-World Arm Use After Stroke

**Author:** Asad NADEEM  
**Course:** Python-R-Git  
**Program:** Master STAPS — Sciences Technologies Mouvement  
**Lab:** EuroMov Digital Health in Motion — University of Montpellier  
**Supervisors:** Karima BAKHTI, Makii MUTHALIB, Nicolas SUTTON-CHARANI  

**GitHub:** https://github.com/asadazizmughal/predicting-arm-use-stroke

---

## Scientific Question

Can clinical motor and functional tests predict how much stroke patients actually use their paretic (weak) arm in daily life at home?

## Notebook overview

1. **Setup** — import libraries, create output folder
2. **Load clinical data** — 30 patients with FMUE, WMFT, BBT, Barthel, PANU scores
3. **Load accelerometry data** — 7-day wrist-worn sensor counts
4. **Compute funcUseRatio** — target variable
5. **Merge datasets**
6. **LOOCV modeling** with in-fold imputation (Prof. Dray's recommendation)
7. **Compare models** — Linear Regression, Ridge, Lasso
8. **Feature importance & discrepancy analysis**

---

## 1. Setup

We import all libraries we will use throughout the notebook and set plot styles. The `results/` folder is created automatically if it doesn't exist — this makes the project portable (works on any computer).

In [1]:
# Library imports for the full ML pipeline
# Data handling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
import openpyxl
import os

# Machine learning (scikit-learn)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

# Plot settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Make sure results folder exists
os.makedirs('results', exist_ok=True)

print("Setup complete — libraries loaded, results folder ready.")

Setup complete — libraries loaded, results folder ready.


## 2. Load Clinical Data

The clinical data comes from the **ReArm protocol** (Muller et al., 2021), a randomized controlled trial at CHU Montpellier. Thirty post-stroke patients were assessed with standardized clinical tests.

We use `openpyxl` (cell-by-cell reading) rather than `pandas.read_excel` because the file has header formatting issues (extra spaces, merged cells) that cause pandas to misread columns.

**Variables loaded:**

- `SUBJID` — patient identifier
- `Sex`, `Age`, `Paresis` — demographic and clinical info
- `PANU` — Proximal Arm Non-Use (%)
- `Barthel` — Barthel Index of daily living independence (/100)
- `FMUE_TOTAL` — Fugl-Meyer Upper Extremity (/66)
- `WMFT_Total` — Wolf Motor Function Test (/75)
- `BBT_P` — Box & Block Test, paretic hand

In [2]:
# Open the Excel workbook
workbook = openpyxl.load_workbook('data/Outcomes_Rearm_01_04_23.xlsx')
sheet = workbook['V1']  # The file has only one sheet

# Read each row into a dictionary, then build a DataFrame
clinical_rows = []
for row_idx in range(2, sheet.max_row + 1):   # row 1 is headers, so start at row 2
    clinical_rows.append({
        'SUBJID':      sheet.cell(row_idx, 1).value,
        'Sex':         sheet.cell(row_idx, 3).value,
        'Age':         sheet.cell(row_idx, 4).value,
        'Paresis':     sheet.cell(row_idx, 6).value,
        'PANU':        sheet.cell(row_idx, 7).value,
        'Barthel':     sheet.cell(row_idx, 8).value,
        'FMUE_TOTAL':  sheet.cell(row_idx, 11).value,
        'WMFT_Total':  sheet.cell(row_idx, 12).value,
        'BBT_P':       sheet.cell(row_idx, 17).value,
    })

clinical_df = pd.DataFrame(clinical_rows)

# Standardize the patient ID so it matches the accelerometry file
# Clinical file uses 'C1-P01', accelerometry uses 'C1P01'
clinical_df['ID'] = clinical_df['SUBJID'].str.replace('-', '')

# Quick check
print(f"Clinical data loaded: {len(clinical_df)} patients")
clinical_df.head()

Clinical data loaded: 30 patients


,SUBJID,Sex,Age,Paresis,PANU,Barthel,FMUE_TOTAL,WMFT_Total,BBT_P,ID
0,C1-P01,Homme,63,Droite,3.0,85.0,45,64,38,C1P01
1,C1-P02,Homme,49,Gauche,3.0,100.0,65,73,52,C1P02
2,C1-P03,Homme,59,Gauche,2.0,90.0,48,56,11,C1P03
3,C1-P04,Homme,61,Gauche,1.0,95.0,60,62,27,C1P04
4,C1-P05,Homme,52,Gauche,3.0,90.0,57,57,15,C1P05


## 3. Load Accelerometry Data

Patients wore **accelerometers on both wrists** for 7 days at home. For each day and each arm, the data contains a count of *functional movements* computed with the algorithm from Leuenberger et al. (2017):

- Forearm amplitude > 30°
- Within ±30° of horizontal  
- In 2-second non-overlapping windows

This filters out passive movements like arm swinging during walking.

The file uses `;` as separator (European CSV convention). For each patient we have 21 daily columns: 7 days × 3 arms (Paretic, Non-Paretic, Bilateral).

In [3]:
# Read accelerometry CSV (semicolon separator — European format)
accel_df = pd.read_csv('data/all_day_FuncUse_uni_bi.csv', sep=';')

print(f"Accelerometry data loaded: {len(accel_df)} patients")
print(f"Number of columns: {accel_df.shape[1]}")

# Check missing values
missing_counts = accel_df.isnull().sum()
print(f"\nColumns with missing values:")
print(missing_counts[missing_counts > 0].to_string())

accel_df.head()

Accelerometry data loaded: 30 patients
Number of columns: 22

Columns with missing values:
Non_Paretic_Day5    1
Paretic_Day5        1
Bilat_Day5          1
Non_Paretic_Day6    1
Paretic_Day6        1
Bilat_Day6          1
Non_Paretic_Day7    3
Paretic_Day7        3
Bilat_Day7          3


,ID,Non_Paretic_Day1,Paretic_Day1,Bilat_Day1,Non_Paretic_Day2,Paretic_Day2,Bilat_Day2,Non_Paretic_Day3,Paretic_Day3,Bilat_Day3,...,Bilat_Day4,Non_Paretic_Day5,Paretic_Day5,Bilat_Day5,Non_Paretic_Day6,Paretic_Day6,Bilat_Day6,Non_Paretic_Day7,Paretic_Day7,Bilat_Day7
0,C1P01,2599,2777,1366,2567,2349,874,3737,3737,1437,...,1293,2780.0,2582.0,988.0,2821.0,3119.0,1027.0,2513.0,2849.0,800.0
1,C1P02,3562,1525,1001,3817,1675,889,3329,1247,573,...,514,3273.0,1480.0,641.0,2725.0,959.0,542.0,3353.0,1086.0,589.0
2,C1P03,4108,306,77,3835,410,85,6198,399,160,...,209,4936.0,227.0,70.0,6751.0,417.0,290.0,5712.0,285.0,101.0
3,C1P04,2055,498,98,3462,510,240,2738,416,97,...,164,3553.0,473.0,181.0,2912.0,549.0,128.0,3232.0,637.0,199.0
4,C1P05,1243,211,137,5050,679,333,6065,806,278,...,275,5048.0,707.0,392.0,5175.0,496.0,409.0,5273.0,592.0,428.0
